In [3]:
import pandas as pd
import json

In [112]:
# JSONL 파일 불러오기
df = pd.read_json("./data/dev/pii_synth_finance_dev.jsonl", lines=True)

# 데이터 확인
# display(df.head())

In [118]:
# df.head()

In [6]:
df['doc_type'].unique()

<StringArray>
[    '공지',   '대출상담',   '민원현황',  '사내메신저',    '회의록',   '업무메일',   '인사발령',
 '고객정보변경',   '상품안내', '카드분실상담', '이상거래탐지',   '연체안내',   '민원처리',  '용역계약서']
Length: 14, dtype: str

In [120]:
with open("./data/dev/pii_synth_finance_dev.jsonl", "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f]


In [122]:
# # 데이터 확인
# print(json.dumps(data[7], indent=4, ensure_ascii=False))

In [124]:
# 검색할 ID
target_id = "D0031"

# ID가 일치하는 데이터 검색
result = next((item for item in data if item.get("id") == target_id), None)

# 결과 출력
if result is not None:
    print(json.dumps(result, indent=4, ensure_ascii=False))
else:
    print("해당 ID를 찾을 수 없습니다.")

{
    "id": "D0031",
    "split": "dev",
    "doc_type": "민원처리",
    "template": "complaint_2",
    "text": "[고객 불만 접수] 자동이체 중복 출금\n접수번호 SB-2026-23395 / 2026년 12월 27일\n\n고객 전서윤님이 이번 달 카드 대금이 온누리은행 계좌(917-282973-24)에서 두 번 출금되었다고 항의함. 담당 김 대리가 거래 내역을 확인한 결과 새봄카드 배치 오류로 중복 출금된 것이 맞음.\n중복 출금액 120,000원은 당일 환급 처리함. 전 서윤 고객은 재발 방지 대책을 요청하였고, IT운영팀 이준우 사원에게 원인 분석을 의뢰함.\n회신 방법: 이메일(sy.jeon68@example.com)",
    "entities": [
        {
            "label": "person",
            "text": "전서윤",
            "start": 61,
            "end": 64,
            "person_id": "P052",
            "form": "full"
        },
        {
            "label": "account_number",
            "text": "917-282973-24",
            "start": 88,
            "end": 101,
            "person_id": "P052"
        },
        {
            "label": "person",
            "text": "전 서윤",
            "start": 198,
            "end": 202,
            "person_id": "P052",
            "form": "spaced"
        },
        {
            "l

In [128]:
# 원하는 문서 조회
doc = df[df["id"] == "D0001"].iloc[0]

print(doc["text"])
print("\n정답 라벨:")
for entity in doc["entities"]:
    print(entity)

[필수] 하반기 개인정보 보호 교육 안내
대상: 전 임직원
기간: 2026년 4월 9일 ~ 2026년 10월 11일

이번 교육에서는 생성형 AI 서비스에 고객 정보를 입력하지 않는 원칙, 메신저로 고객 연락처를 주고받지 않는 원칙을 다룹니다. 미이수 시 인사 평가에 반영될 수 있습니다.
교육 문의: 정보보호팀 (내선 5448)

정답 라벨:


In [ ]:
# for _, row in df.drop_duplicates("doc_type").iterrows():
#     print(f"\n===== {row['doc_type']} =====")
#     print(row["text"])

In [11]:
# docs = df.loc[df["doc_type"] == "업무메일"].to_dict(orient="records")

# for doc in docs:
#     print(json.dumps(doc, ensure_ascii=False, indent=4))
#     print("-" * 80)

# print(f"총 {len(docs)}건")

## 업무메일

In [47]:

mail = df[df["doc_type"] == "업무메일"].copy()

print("문서 수:", len(mail))
print(mail["template"].value_counts())

문서 수: 12
template
email_internal    6
email_audit       3
email_vendor      3
Name: count, dtype: int64


In [13]:
#templete 구조 확인
for template, group in mail.groupby("template"):
    print(f"\n=== {template} ===")
    print(group.iloc[0]["text"])


=== email_audit ===
제목: 개인정보 처리 위탁 점검 결과 공유

정보보호팀 정서현입니다.
코드브릿지에 대한 상반기 개인정보 처리 위탁 점검 결과를 공유드립니다.

1. 점검 일시: 2026-02-04
2. 점검자: 정서현, 이서연
3. 수탁사 담당자: 문은정 매니저 (010.0561.7253, ej.moon49@codebridge.example.com)
4. 주요 지적 사항: 상담 녹취 파일 보관 기간 초과, 퇴사자 계정 미삭제 2건

수탁사에는 2026년 9월 6일까지 개선 계획 제출을 요청했습니다. 관련 문의는 이 서연 과장에게 해주세요.

=== email_internal ===
제목: 고객 민원 회신 초안 검토 부탁드립니다

지영 씨, 안녕하세요.
어제 접수된 민원 건(202625845497) 회신 초안 보내드립니다. 민원인 정하은 고객 주소가 경기도 성남시 분당구 다솜길 33, 13층으로 되어 있는데, 지난달 변경 신청 내역과 다른 것 같아 확인 부탁드려요.
검토 후 허수아 부장님께 바로 올려 주셔도 됩니다.

김은정 드림

=== email_vendor ===
제목: [새봄카드] 차세대 카드 시스템 테스트 일정 협의

코드브릿지 손하은 책임님께,

안녕하세요, 새봄카드 IT운영팀 최혜진입니다.
지난 회의에서 말씀드린 통합 테스트 일정을 다음과 같이 제안드립니다.

- 1차 테스트: 2026년 10월 24일
- 2차 테스트: 2026-12-24

테스트 계정은 보안 정책상 메일로 보내드릴 수 없어, 저희 정보보호팀 박하린 과장이 별도로 전달드릴 예정입니다.
일정 확인 후 회신 부탁드리며, 급한 건은 제 휴대전화 010-0282-9512로 연락 주세요.

최혜진 드림
IT운영팀 | 02-3961-1452 | hj.choi34@saebomcard.example.com


In [14]:
#개인정보 라벨 확인
# entities를 행 단위로 분리
ent = mail[["id", "template", "entities"]].explode("entities")

ent = pd.concat([
    ent.drop(columns="entities").reset_index(drop=True),
    pd.json_normalize(ent["entities"]).reset_index(drop=True)
], axis=1)

# 라벨별 분포
print(ent["label"].value_counts())

# 이름 표기 형태
print(ent[ent["label"] == "person"]["form"].value_counts())

label
person          51
phone_number     9
address          6
email            6
Name: count, dtype: int64
form
full      42
given      6
spaced     3
Name: count, dtype: int64


In [15]:
#hard_case 확인
# hard_cases 종류와 빈도
print(mail["hard_cases"].explode().value_counts())

# 특정 어려운 사례에 해당하는 문서 확인
mail[mail["hard_cases"].apply(lambda x: "homonym" in x)][
    ["id", "template", "hard_cases"]
]

hard_cases
homonym                 7
given_only              6
spaced_name             3
phone_format_variant    2
Name: count, dtype: int64


,id,template,hard_cases
13,D0014,email_internal,"[given_only, homonym]"
31,D0032,email_audit,"[homonym, phone_format_variant, spaced_name]"
61,D0062,email_audit,"[homonym, spaced_name]"
85,D0086,email_vendor,"[homonym, phone_format_variant]"
95,D0096,email_audit,"[homonym, spaced_name]"
96,D0097,email_internal,"[given_only, homonym]"
99,D0100,email_internal,"[given_only, homonym]"


In [126]:
# # 원하는 문서 조회
# doc = mail[mail["id"] == "D0014"].iloc[0]

# print(doc["text"])
# print("\n정답 라벨:")
# for entity in doc["entities"]:
#     print(entity)

In [104]:
doc_email = df.loc[
    (df["doc_type"] == "업무메일") & (df["template"] == "email_internal")
].iloc[5]

# print(json.dumps(doc_email, ensure_ascii=False, indent=4))
print(doc_email["text"])

제목: 고객 민원 회신 초안 검토 부탁드립니다

민준 씨, 안녕하세요.
어제 접수된 민원 건(202622109438) 회신 초안 보내드립니다. 민원인 이선우 고객 주소가 인천광역시 부평구 새봄로 24, 101동 602호로 되어 있는데, 지난달 변경 신청 내역과 다른 것 같아 확인 부탁드려요.
검토 후 남성호 부장님께 바로 올려 주셔도 됩니다.

조민준 드림


In [44]:
#어려운 사례 문서별로 정리
# 검토할 문서
target_ids = ["D0014", "D0032"]

for doc_id in target_ids:
    doc = mail[mail["id"] == doc_id].iloc[0]

    print(f"\n=== {doc_id} / {doc['template']} ===")

    for e in doc["entities"]:
        start, end = e["start"], e["end"]

        context = doc["text"][max(0, start-15):end+15]

        print(e["label"], ":", repr(context))


=== D0014 / email_internal ===
person : ' 초안 검토 부탁드립니다\n\n지영 씨, 안녕하세요.\n어제 접'
person : '초안 보내드립니다. 민원인 정하은 고객 주소가 경기도 성남시'
address : '민원인 정하은 고객 주소가 경기도 성남시 분당구 다솜길 33, 13층으로 되어 있는데, 지난달 '
person : '확인 부탁드려요.\n검토 후 허수아 부장님께 바로 올려 주셔도'
person : '로 올려 주셔도 됩니다.\n\n김은정 드림'

=== D0032 / email_audit ===
person : '검 결과 공유\n\n정보보호팀 정서현입니다.\n코드브릿지에 대한 '
person : '-02-04\n2. 점검자: 정서현, 이서연\n3. 수탁사 담당'
person : '4\n2. 점검자: 정서현, 이서연\n3. 수탁사 담당자: 문은'
person : '서연\n3. 수탁사 담당자: 문은정 매니저 (010.0561.'
phone_number : ' 담당자: 문은정 매니저 (010.0561.7253, ej.moon49@cod'
email : '010.0561.7253, ej.moon49@codebridge.example.com)\n4. 주요 지적 사항: '
person : '요청했습니다. 관련 문의는 이 서연 과장에게 해주세요.'


In [ ]:
# #개인정보 나타나는 위치 정량 분석
# # 개인정보 위치 계산
# ent = mail[["id", "template", "text", "entities"]].explode("entities").reset_index(drop=True)
# ent = ent[ent["entities"].notna()].copy()

# ent["label"] = ent["entities"].apply(lambda x: x["label"])

# # 문서 길이 대비 개인정보 시작 위치
# ent["position"] = ent.apply(
#     lambda x: x["entities"]["start"] / len(x["text"]),
#     axis=1
# )

# # 문서를 4개 구간으로 분할
# ent["section"] = pd.cut(
#     ent["position"],
#     bins=[0, 0.25, 0.5, 0.75, 1],
#     labels=["0~25%", "25~50%", "50~75%", "75~100%"],
#     include_lowest=True
# )

# # 템플릿별 개인정보 위치
# pd.crosstab(
#     [ent["template"], ent["label"]],
#     ent["section"]
# ).to_clipboard()

In [ ]:
# # 1. 이메일별 라벨 조합 생성 (중복 제거)
# mail["label_combo"] = mail["entities"].apply(
#     lambda entities: "[" + ", ".join(
#         sorted(
#             {e["label"] for e in entities},
#             key=lambda x: (x != "person", x)
#         )
#     ) + "]"
# )

# # 2. 템플릿별 라벨 조합 분포
# result = (
#     mail.groupby(["template", "label_combo"])
#     .size()
#     .reset_index(name="문서 수")
# ).to_clipboard()

# # display(result)

In [113]:
contract = df[df["doc_type"] == "사내메신저"].copy()

print("문서 수:", len(contract))
print(contract["template"].value_counts())

# 문서별 원문 확인
for _, doc in contract.iterrows():
    print(f"\n=== {doc['id']} ===")
    print(doc["text"])

문서 수: 2
template
messenger    2
Name: count, dtype: int64

=== D0007 ===
[사내 메신저] 2026-08-04
김서연: 승현아, 오늘 회의 자료 어디 있어?
신승현: 공유 폴더에 올려놨어. 근데 전서윤 고객 건은 아직 정리 못 했어
김서연: 그 고객 연락처 010-0863-1041 맞지? 내가 오후에 전화해 볼게
신승현: 응 부탁해. 아 그리고 허 부장님이 내일까지 보고서 달래
김서연: 알겠어. 은서랑 같이 정리해서 올릴게

=== D0009 ===
[사내 메신저] 2026년 3월 13일
이윤서: 지영아, 오늘 회의 자료 어디 있어?
최지영: 공유 폴더에 올려놨어. 근데 정하은 고객 건은 아직 정리 못 했어
이윤서: 그 고객 연락처 010-0922-9038 맞지? 내가 오후에 전화해 볼게
최지영: 응 부탁해. 아 그리고 허 부장님이 내일까지 보고서 달래
이윤서: 알겠어. 소율이랑 같이 정리해서 올릴게


In [116]:
# # 이상거래탐지 문서 추출
# fds = df[df["doc_type"] == "이상거래탐지"].copy()

# # 템플릿별 라벨 조합
# fds["label_combo"] = fds["entities"].apply(
#     lambda entities: tuple(sorted(set(e["label"] for e in entities)))
# )

# display(
#     fds.groupby(["template", "label_combo"])
#     .size()
#     .reset_index(name="문서 수")
# )

In [115]:
# for _, doc in fds.iterrows():
#     print(f"\n=== {doc['id']} / {doc['template']} ===")
#     print(doc["text"])
#     print("-" * 70)

In [114]:
hr = df[df["doc_type"] == "사내메신저"].copy()

for _, doc in hr.iterrows():
    print(f"\n=== {doc['id']} / {doc['template']} ===")
    print(doc["text"])

    print("\n[정답 라벨]")
    for e in doc["entities"]:
        print(f"{e['label']}: {e['text']}")

    print("\n[Hard Cases]", doc["hard_cases"])
    print("=" * 60)


=== D0007 / messenger ===
[사내 메신저] 2026-08-04
김서연: 승현아, 오늘 회의 자료 어디 있어?
신승현: 공유 폴더에 올려놨어. 근데 전서윤 고객 건은 아직 정리 못 했어
김서연: 그 고객 연락처 010-0863-1041 맞지? 내가 오후에 전화해 볼게
신승현: 응 부탁해. 아 그리고 허 부장님이 내일까지 보고서 달래
김서연: 알겠어. 은서랑 같이 정리해서 올릴게

[정답 라벨]
person: 김서연
person: 승현
person: 신승현
person: 전서윤
person: 김서연
phone_number: 010-0863-1041
person: 신승현
person: 김서연
person: 은서

[Hard Cases] ['given_only', 'josa_attached', 'surname_title_unlabeled']

=== D0009 / messenger ===
[사내 메신저] 2026년 3월 13일
이윤서: 지영아, 오늘 회의 자료 어디 있어?
최지영: 공유 폴더에 올려놨어. 근데 정하은 고객 건은 아직 정리 못 했어
이윤서: 그 고객 연락처 010-0922-9038 맞지? 내가 오후에 전화해 볼게
최지영: 응 부탁해. 아 그리고 허 부장님이 내일까지 보고서 달래
이윤서: 알겠어. 소율이랑 같이 정리해서 올릴게

[정답 라벨]
person: 이윤서
person: 지영
person: 최지영
person: 정하은
person: 이윤서
phone_number: 010-0922-9038
person: 최지영
person: 이윤서
person: 소율

[Hard Cases] ['given_only', 'homonym', 'josa_attached', 'surname_title_unlabeled']
